# Fine-tuning com LoRA — PT → Glosa em LIBRAS

TCC — MBA em Data Science (UNIFOR). Este notebook treina o terceiro dos três modelos comparados no trabalho: **fine-tuning de um modelo pré-treinado (PTT5-small) via LoRA**, sobre o corpus VLibrasBD.

**Antes de rodar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → GPU (**T4** é suficiente).

**Ordem das células:** rode de cima pra baixo, sem pular. A célula de treino pode levar de 1 a 3 horas dependendo do tamanho efetivo dos dados — o notebook já vem com early stopping, então ele para sozinho quando parar de melhorar.

## 1. Confirma que a GPU foi alocada

In [ ]:
!nvidia-smi

Mon Aug 24 12:35:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Se aparecer `NVIDIA-SMI has failed` ou nenhuma GPU listada, volta em `Ambiente de execução` → `Alterar tipo de ambiente de execução` e seleciona GPU antes de continuar.

## 2. Instala as dependências

In [ ]:
!pip install -q transformers peft accelerate sentencepiece datasets sacrebleu

## 3. Monta o Google Drive

Usamos o Drive para persistir os dados e, principalmente, os checkpoints — se o Colab desconectar no meio do treino (acontece com a versão gratuita), o progresso salvo no Drive não se perde.

Na primeira vez, vai pedir autorização — segue o link, autoriza com a sua conta Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/libras-tcc'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
print(f'Pasta do projeto no Drive: {PROJECT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta do projeto no Drive: /content/drive/MyDrive/libras-tcc


## 4. Envia os dados processados

Precisa dos três arquivos que o `data_prep.py` já gerou no seu PC: `train.csv`, `val.csv`, `test.csv` (pasta `data/processed/` do projeto).

**Se já subiu esses arquivos pro Drive antes**, pula esta célula — eles continuam lá entre sessões. Confirma com a célula de verificação logo abaixo.

**Se é a primeira vez**, roda a célula abaixo e seleciona os três arquivos (`train.csv`, `val.csv`, `test.csv`) juntos na janela de upload.

In [ ]:
import os

data_dir = f'{PROJECT_DIR}/data'
existing = [f for f in ['train.csv', 'val.csv', 'test.csv'] if os.path.exists(f'{data_dir}/{f}')]

if len(existing) == 3:
    print('Os três arquivos já estão no Drive — pode pular a célula de upload.')
else:
    print(f'Faltam arquivos no Drive (encontrados: {existing}). Rode a célula de upload abaixo.')

Os três arquivos já estão no Drive — pode pular a célula de upload.


In [ ]:
# Só rode esta célula se a verificação acima disse que faltam arquivos.
from google.colab import files
import shutil

uploaded = files.upload()  # seleciona train.csv, val.csv e test.csv juntos
for fname in uploaded:
    shutil.move(fname, f'{PROJECT_DIR}/data/{fname}')
print('Arquivos movidos para', f'{PROJECT_DIR}/data')

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv
Arquivos movidos para /content/drive/MyDrive/libras-tcc/data


## 5. Configuração

Mesmos hiperparâmetros documentados em `src/finetune_lora.py` do projeto — mantém os dois sincronizados se ajustar algo aqui.

In [ ]:
MODEL_NAME = "unicamp-dl/ptt5-small-portuguese-vocab"
MAX_SOURCE_LEN = 64
MAX_TARGET_LEN = 64

# Hiperparâmetros do LoRA — documentar na Metodologia do TCC
LORA_R = 16
LORA_ALPHA = 32  # convenção: alpha = 2 * r
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q", "v"]  # projeções de atenção query/value

EPOCHS = 15          # teto de segurança — early stopping deve parar antes
EARLY_STOPPING_PATIENCE = 3

# Opcional: reduz o treino se as épocas estiverem demorando demais.
# None = usa os 101.850 pares completos.
TRAIN_MAX_EXAMPLES = None

TASK_PREFIX = "traduzir para glosa em libras: "

ADAPTER_DIR = f'{PROJECT_DIR}/models/lora_adapter'
CHECKPOINTS_DIR = f'{PROJECT_DIR}/models/lora_checkpoints'

## 6. Carrega o modelo base e aplica o LoRA

In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 22.6 MB/s eta 0:00:00


In [ ]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # confirma a fração de parâmetros treináveis — vale citar no texto

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


## 7. Carrega e tokeniza os dados

In [ ]:
import pandas as pd
from datasets import Dataset

def load_split(name):
    df = pd.read_csv(f'{PROJECT_DIR}/data/{name}.csv')
    if name == 'train' and TRAIN_MAX_EXAMPLES is not None and len(df) > TRAIN_MAX_EXAMPLES:
        df = df.sample(TRAIN_MAX_EXAMPLES, random_state=42).reset_index(drop=True)
    return Dataset.from_pandas(df[['pt', 'glosa']])

train_ds, val_ds = load_split('train'), load_split('val')
print(f'Treino: {len(train_ds)} pares | Validação: {len(val_ds)} pares')

def preprocess(batch):
    inputs = [TASK_PREFIX + str(x) for x in batch['pt']]
    targets = [str(x) for x in batch['glosa']]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=['pt', 'glosa'])
val_ds = val_ds.map(preprocess, batched=True, remove_columns=['pt', 'glosa'])

Treino: 101850 pares | Validação: 12731 pares


Map:   0%|          | 0/101850 [00:00<?, ? examples/s]

Map:   0%|          | 0/12731 [00:00<?, ? examples/s]

## 8. Treino

Com early stopping ativo (`EARLY_STOPPING_PATIENCE=3`): para sozinho se `eval_loss` não melhorar por 3 épocas seguidas, e recupera automaticamente o melhor checkpoint (`load_best_model_at_end=True`).

Os checkpoints salvam no Drive a cada época — se o Colab desconectar, o progresso não se perde (mas você precisa rodar as células de 1 a 7 de novo pra recriar o ambiente, e então rodar `trainer.train(resume_from_checkpoint=True)` em vez de `trainer.train()` — comentado abaixo caso precise).

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=EPOCHS,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

trainer.train()
# Se precisar retomar de um checkpoint salvo após desconexão:
# trainer.train(resume_from_checkpoint=True)

Epoch,Training Loss,Validation Loss
1,5.125125,2.208382
2,4.824797,2.052962
3,4.718230,1.967006
4,4.489025,1.913293
5,4.369214,1.867260
6,4.308468,1.829248
7,4.310638,1.798276
8,4.232256,1.775977
9,4.225377,1.756968
10,4.104822,1.740584


Epoch,Training Loss,Validation Loss
1,5.125125,2.208382
2,4.824797,2.052962
3,4.718230,1.967006
4,4.489025,1.913293
5,4.369214,1.867260
6,4.308468,1.829248
7,4.310638,1.798276
8,4.232256,1.775977
9,4.225377,1.756968
10,4.104822,1.740584


TrainOutput(global_step=47745, training_loss=4.438645498020285, metrics={'train_runtime': 9189.081, 'train_samples_per_second': 166.257, 'train_steps_per_second': 5.196, 'total_flos': 2.3668947861504e+16, 'train_loss': 4.438645498020285, 'epoch': 15.0})

## 9. Salva o histórico de treino (curva de aprendizado)

Mesmo formato usado pelo seq2seq do zero — dá pra comparar as duas curvas lado a lado na Metodologia.

In [10]:
history_rows = []
for entry in trainer.state.log_history:
    if 'eval_loss' in entry:
        history_rows.append({'epoch': round(entry.get('epoch', 0), 2), 'eval_loss': entry['eval_loss']})
    elif 'loss' in entry:
        history_rows.append({'epoch': round(entry.get('epoch', 0), 2), 'train_loss': entry['loss']})

history_df = pd.DataFrame(history_rows)
history_path = f'{PROJECT_DIR}/models/lora_training_history.csv'
history_df.to_csv(history_path, index=False)
print(f'Salvo em {history_path}')
history_df.tail(10)

Salvo em /content/drive/MyDrive/libras-tcc/models/lora_training_history.csv


,epoch,train_loss,eval_loss
959,14.86,3.998853,NaN
960,14.88,4.072795,NaN
961,14.89,4.110445,NaN
962,14.91,4.137130,NaN
963,14.92,4.149518,NaN
964,14.94,4.085041,NaN
965,14.95,4.089936,NaN
966,14.97,4.082374,NaN
967,14.99,4.093990,NaN
968,15.00,NaN,1.701195


## 10. Salva o adapter LoRA

Só as matrizes LoRA são salvas (não o modelo base inteiro) — arquivo pequeno, alguns MB, fácil de baixar e versionar no GitHub do projeto.

In [11]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter salvo em {ADAPTER_DIR}')

!du -sh {ADAPTER_DIR}

Adapter salvo em /content/drive/MyDrive/libras-tcc/models/lora_adapter
4.6M	/content/drive/MyDrive/libras-tcc/models/lora_adapter


## 11. Teste rápido de sanidade

Confirma que o modelo treinado gera glosa antes de sair da sessão.

In [12]:
model.eval()
test_sentences = [
    'Maria comprou um carro novo.',
    'Vocês receberam a carta?',
    'Qualquer cidadão pode solicitar o documento.',
]
for s in test_sentences:
    inputs = tokenizer(TASK_PREFIX + s, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    print(f'{s}\n  -> {tokenizer.decode(output_ids[0], skip_special_tokens=True)}\n')

Maria comprou um carro novo.
  -> MARIA ENVIAR FORMA&DOCUMENTO [PONTO]

Vocês receberam a carta?
  -> VOCÊS RECEBER [INTERROGAÇÃO]

Qualquer cidadão pode solicitar o documento.
  -> PODER&POSSIBILIDADE PODER&POSSIBILIDADE PODER&POSSIBILIDADE PODER&POSSIBILIDADE [PONTO]



## 12. Baixa o adapter para o projeto local

Compacta e baixa direto — depois é só extrair dentro de `models/lora_adapter/` no seu projeto local (WSL/Windows), para o `evaluate.py` e o `poc_pipeline.py` conseguirem usá-lo.

In [13]:
import shutil
from google.colab import files

zip_path = '/content/lora_adapter'
shutil.make_archive(zip_path, 'zip', ADAPTER_DIR)
files.download(f'{zip_path}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Próximo passo

No seu projeto local:

1. Extrai o `lora_adapter.zip` baixado dentro de `models/lora_adapter/` (crie a pasta se não existir)
2. Também baixa `lora_training_history.csv` do Drive (`MyDrive/libras-tcc/models/`) e coloca em `models/` no projeto local
3. Roda `python src/evaluate.py` — agora com os três modelos (regras, seq2seq, LoRA) ele gera a comparação completa: tabela geral, por categoria linguística e por domínio
4. Depois, `python src/poc_pipeline.py caminho/para/audio.mp3` para a prova de conceito ponta a ponta